# Entity Ruler - rule based matching

This notebook explores a rule based skill extraction with Entity Ruler.

In [1]:
import json
import pandas as pd
import textwrap

from src.skills import TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM
from src.models import build_nlp, entity_ruler_extract_skills

In [2]:
with open("../data/train_set.json", "r", encoding="utf-8") as file:
    train_set = json.load(file)

train_set_df = pd.DataFrame(train_set)
train_set_df.head(3)

,id,title,employer,municipality,region,occupation,published,description
0,31266388,Chefsarkitekt – samla ihop och utveckla vår en...,REGERINGSKANSLIET,Stockholm,Stockholms län,Enterprise Architect,2026-07-10T15:15:55,"Digitaliseringsavdelningen, Sektionen för Stra..."
1,30760800,Senior Solutions Architect SAP,Cepheid AB,Solna,Stockholms län,Systemarkitekt,2026-03-18T00:06:15,"At Cepheid, one of Danaher’s 15+ operating com..."
2,30770777,Ingenjör med inriktning mot tillgänglighetsanalys,Försvarets Materielverk,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2026-03-19T13:11:24,Har du erfarenhet av analysarbete och komplexa...


In [3]:
# Building pipeline
nlp = build_nlp()

ruler = nlp.get_pipe("entity_ruler")

# Show how many patterns
print(len(ruler.patterns), "patterns")

633 patterns


The example shows that the rules catch strictly what is written in the skill dictionaries and that is a strong limit due to the context and the inflection of expressions:  
**enterprisearkitektur, kravanalys, integrationsarkitektur, informationsmodeller, molnplattformar, molnstrategier, multi-cloud, dataplattformar**  
are hard skills, but my dictionaries are limited to other forms of these expressions and therefore miss them.

In [4]:
# Checking on an example - parts taken from training set
ad = "Vi moderniserar vår enterprisearkitektur, inför generativ AI, skalar upp molnplattformar och bygger en datadriven organisation. " \
"Det är stort, komplext och samhällsviktigt. " \
"Nu söker vi en chefsarkitekt som vill ta ett helhetsgrepp om arkitekturen och leda riktningen framåt. " \
"Som chefsarkitekt har du en ledande roll, men är inte linjechef. " \
"Ditt uppdrag: Ha huvudansvaret för all arkitektur – verksamhets-, informations-, applikations- och teknisk arkitektur. " \
"Leda och samordna EA‑arbetet, säkerställa struktur, riktning och kvalitet. " \
"Säkerställa att arkitekturen stödjer verksamhetens mål, strategier och förändringsinitiativ. Driva fram gemensamma principer, riktlinjer och målarkitekturer. " \
"Erfarenhet av verksamhetsdriven utveckling och kravanalys. " \
"God förståelse för moderna tekniska plattformar, integrationsarkitektur och informationsmodeller. " \
"Kunskap om arkitekturmetoder som TOGAF, ArchiMate eller liknande. " \
"Erfarenhet av AI-relaterad arkitektur, dataplattformar eller molnstrategier. " \
"Erfarenhet av hybrid- eller multi-cloudlösningar."

print(entity_ruler_extract_skills(ad, nlp))

['ArchiMate', 'Artificial Intelligence', 'TOGAF-certifiering']


## The Entity Ruler pipeline on the entire training set

In [5]:
train_set_df["skills"] = train_set_df["description"].apply(lambda text:entity_ruler_extract_skills(text, nlp))

In [6]:
train_set_df[["description", "skills"]].head(10)

,description,skills
0,"Digitaliseringsavdelningen, Sektionen för Stra...","[ArchiMate, Artificial Intelligence, English, ..."
1,"At Cepheid, one of Danaher’s 15+ operating com...","[Kanban, SAP]"
2,Har du erfarenhet av analysarbete och komplexa...,"[English, Swedish]"
3,About Justera Group Justera Group AB is a lead...,"[Amazon Web Services, Automation, English, Swe..."
4,Vill du ta en ledande roll i att forma vår fra...,[]
5,About the Company Avaron helps you find assign...,[]
6,"At Ubiquiti Inc., we create technology platfor...",[DevOps]
7,"Myndigheten för civilt försvar är den ledande,...","[English, RAN, Swedish]"
8,Är du en av de vassaste på KTH som tar examen ...,"[Artificial Intelligence, Github, Machine lear..."
9,Om företaget Avaron AB är ett växande konsultb...,"[Data modeling, English, Kubernetes, Quality A..."


In [7]:
# How many skills per add?
train_set_df["n_skills"] = train_set_df["skills"].str.len()
print(train_set_df["n_skills"].describe())
print()

# Most common skills across all ads
skill_counts = train_set_df["skills"].explode().value_counts()
print(skill_counts.head(30))

count    2606.000000
mean        3.828473
std         2.713747
min         0.000000
25%         2.000000
50%         4.000000
75%         5.000000
max        18.000000
Name: n_skills, dtype: float64

skills
Swedish                                      1221
English                                       833
Artificial Intelligence                       485
Automation                                    369
Windows Azure                                 332
Digitalisering                                314
Leadership                                    293
Agile                                         255
Python                                        249
SQL                                           249
Machine learning                              224
General Data Protection Regulation (GDPR)     194
Kravhantering                                 184
SAP                                           181
Verksamhetsutveckling                         177
Driva workshops                            

### Checking empty matches

In some cases Entity Ruler could not match any of its patterns to the ad descriptions. The main causes are:  
- Some ads are simply too short and don't contain any real hard skill (example ad 30696991).  

- The actual hard skills are not listed in the rules (exampla add 31382042, with Verification & Validation (V&V/VVL)  

- The rules are not flexible enough to catch loose skill patterns in the unstructured text ()example ad 30558951)  


In [8]:
empty_matches = train_set_df.loc[
    train_set_df["skills"].apply(len) == 0,
    ["id", "description", "skills"]
]

print(f"{len(empty_matches)} of {len(train_set_df)} ads have no matches")
empty_matches.head()

235 of 2606 ads have no matches


,id,description,skills
4,30558951,Vill du ta en ledande roll i att forma vår fra...,[]
5,30959653,About the Company Avaron helps you find assign...,[]
10,30696991,About the Company Avaron AB is a growing consu...,[]
18,31382042,Om företaget Hos Avaron får du tryggheten i en...,[]
29,30995743,We are looking for an experienced Requirement ...,[]


In [11]:
def show_ad(text, width=100):
    for paragraph in text.splitlines():
        print(textwrap.fill(paragraph, width=width) if paragraph.strip() else "")

show_ad(empty_matches.iloc[2]["description"])

About the Company Avaron AB is a growing consultancy that matches your expertise with the market's
most interesting assignments, offering a platform where your professional development is central.
About the Assignment You will join a next-generation intranet program within Strategy &
Communications, focused on creating a modern, personalised and scalable digital workplace that
strengthens the employee experience.


In [10]:
# Checks characters before and after a specific detected skill
skill = "Digitalisering"
skill_ads = train_set_df[train_set_df["skills"].apply(lambda skills: skill in skills)]

for _, row in skill_ads.head(5).iterrows():
    print("TITLE:", row["title"])
    text = row["description"]

    for ent in nlp(text).ents:
        if ent.ent_id_ == skill:
            start = ent.start_char
            print(ent.text, "->", ent.ent_id_)
            print(text[max(0, start - 50): start + 50])
            print("-" * 80)

TITLE: Enterprise Architect  Technology Modernization & Innovation
digital transformation -> Digitalisering
ions that directly shape our future and drive our digital transformation. Your mission As Enterprise
--------------------------------------------------------------------------------
TITLE: Senior Ekonomikonsult Microsoft Dynamics 365 Business Central
digitalisering -> Digitalisering
itala samarbetsverktyg kan vi erbjuda en komplett digitalisering av kundernas kärnverksamhet. Fellow
--------------------------------------------------------------------------------
TITLE: IT-arkitekt till Höglandets IT
digital transformation -> Digitalisering
 långsiktiga teknikval, hållbara arkitekturer och digital transformation bidrar du till att skapa fö
--------------------------------------------------------------------------------
digitalisering -> Digitalisering
likvärdig. Du har erfarenhet av arkitekturarbete, digitalisering och komplexa IT-miljöer där du bidr
-------------------------------